# End-to-End PICO Extraction Baseline 
This notebook implements an improved end-to-end extraction approach using:
- Sentence-level splitting and scoring
- Field-specific keyword matching
- Intelligent sentence selection based on relevance
- Fallback heuristics for robustness

In [1]:
import re
import numpy as np
import pandas as pd
import spacy
from sklearn.metrics import precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings('ignore')

# Load spacy model
nlp = spacy.load("en_core_web_sm")

## 1. Define Enhanced Patterns and Keyword Sets

In [2]:
# More specific patterns for each PICO field
POPULATION_KEYWORDS = [
    r'(?:^\s*)?(\d+)\s+(?:patients?|participants?|subjects?|individuals?|people)',
    r'(?:adults?|children|elderly|infants?|neonates?|adolescents?|women|men)\s+(?:with|who|aged|years)',
    r'(?:age[sd]?)\s+(?:\d+[-–]\d+|mean|median|range)',
]

INTERVENTION_KEYWORDS = [
    r'(?:received|treated|administered|given|treated with)\s+([^.;,]*(?:mg|drug|therapy|treatment|intervention).*?)(?:[.;,])',
    r'(?:intervention|treatment|drug|medication|therapy)[:\s]+([^.;,]*?)(?:[.;,])',
    r'(?:group|arm|cohort)\s+[a-z].*?(?:received|given)\s+([^.;,]*?)(?:[.;,])',
]

COMPARATOR_KEYWORDS = [
    r'(?:vs\.?|versus|compared with|compared to|control|placebo)',
    r'(?:control group|standard care|usual care|standard treatment)',
]

OUTCOME_KEYWORDS = [
    r'(?:primary outcome|secondary outcome|primary|secondary)[:\s]*([^.;]*(?:reduced|improved|increased|decreased|change).*?)(?:[.;])',
    r'(?:showed|demonstrated|resulted in|led to)\s+([^.;]*(?:improvement|reduction|increase|decrease).*?)(?:[.;])',
    r'(?:reduced|improved|increased|decreased|change in)\s+([^.;]*?)(?:[.;]|$)',
]

## 2. Sentence-Level Scoring System

In [3]:
class SentenceScorer:
    """Score sentences for relevance to PICO fields"""
    
    def __init__(self):
        self.pop_keywords = ['patient', 'participant', 'subject', 'population', 'enrolled', 'included', 'age', 'demographic']
        self.int_keywords = ['treatment', 'intervention', 'received', 'administered', 'drug', 'therapy', 'mg', 'dose']
        self.comp_keywords = ['versus', 'vs', 'compared', 'control', 'placebo', 'standard']
        self.out_keywords = ['outcome', 'primary', 'secondary', 'improved', 'reduced', 'increased', 'decreased', 'change']
    
    def score_for_field(self, sentence, field_type):
        """Score sentence relevance to a specific PICO field"""
        sentence_lower = sentence.lower()
        score = 0.0
        
        if field_type == 'population':
            for kw in self.pop_keywords:
                if kw in sentence_lower:
                    score += 1.0
            # Boost if contains numbers (likely sample size)
            if re.search(r'\d+', sentence):
                score += 2.0
                
        elif field_type == 'intervention':
            for kw in self.int_keywords:
                if kw in sentence_lower:
                    score += 1.0
            # Boost if contains medical terminology
            if re.search(r'\d+\s*(?:mg|ml|units?)\b', sentence):
                score += 2.0
                
        elif field_type == 'comparator':
            for kw in self.comp_keywords:
                if kw in sentence_lower:
                    score += 2.0
                    
        elif field_type == 'outcome':
            for kw in self.out_keywords:
                if kw in sentence_lower:
                    score += 1.0
            # Boost if contains measurable changes
            if any(x in sentence_lower for x in ['improvement', 'reduction', 'increase', 'decrease']):
                score += 2.0
        
        return score


scorer = SentenceScorer()

## 3. Core Extraction Functions

In [4]:
def split_into_sentences(text):
    """Split text into sentences using spacy"""
    doc = nlp(text)
    return [sent.text.strip() for sent in doc.sents]


def extract_field_from_sentences(sentences, field_type, patterns):
    """Extract field using sentences and patterns"""
    
    # Score all sentences for relevance
    scored_sentences = []
    for sent in sentences:
        score = scorer.score_for_field(sent, field_type)
        if score > 0:
            scored_sentences.append((sent, score))
    
    # Sort by relevance
    scored_sentences.sort(key=lambda x: x[1], reverse=True)
    
    # Try to extract from high-scoring sentences
    for sent, _ in scored_sentences[:3]:  # Check top 3 relevant sentences
        for pattern in patterns:
            match = re.search(pattern, sent, re.I)
            if match:
                if match.groups():
                    extracted = match.group(1).strip()
                else:
                    extracted = match.group(0).strip()
                
                # Clean up
                extracted = re.sub(r'\s+', ' ', extracted)
                if len(extracted) > 5 and len(extracted) < 500:
                    return extracted
    
    return None

In [5]:
def extract_population_end_to_end(text, sentences):
    """Extract population information"""
    patterns = POPULATION_KEYWORDS
    
    # Try sentence-level extraction first
    result = extract_field_from_sentences(sentences, 'population', patterns)
    if result:
        return result
    
    # Fallback: search whole text for patient counts
    match = re.search(r'(\d+)\s+(?:patients?|participants?|subjects?)', text, re.I)
    if match:
        # Return the sentence containing this match
        for sent in sentences:
            if match.group(0) in sent:
                return sent[:150]
    
    return None


def extract_intervention_end_to_end(text, sentences):
    """Extract intervention information"""
    patterns = INTERVENTION_KEYWORDS
    
    result = extract_field_from_sentences(sentences, 'intervention', patterns)
    if result:
        return result
    
    # Fallback: look for treatment/drug mentions
    match = re.search(r'(.*?(?:treatment|drug|therapy|intervention).*?)(?:[.;])', text, re.I | re.S)
    if match:
        text_chunk = match.group(1)
        if len(text_chunk) < 500:
            return text_chunk.strip()
    
    return None


def extract_comparator_end_to_end(text, sentences):
    """Extract comparator/control information"""
    patterns = COMPARATOR_KEYWORDS
    
    result = extract_field_from_sentences(sentences, 'comparator', patterns)
    if result:
        return result
    
    # Fallback: search for comparison keywords
    for comp_kw in ['versus', 'vs.', 'vs', 'compared with', 'compared to', 'placebo', 'control']:
        if comp_kw in text.lower():
            for sent in sentences:
                if comp_kw in sent.lower():
                    return sent[:200]
    
    return None


def extract_outcome_end_to_end(text, sentences):
    """Extract outcome information"""
    patterns = OUTCOME_KEYWORDS
    
    result = extract_field_from_sentences(sentences, 'outcome', patterns)
    if result:
        return result
    
    # Fallback: look for result/finding sentences
    for sent in sentences:
        if any(kw in sent.lower() for kw in ['result', 'outcome', 'found', 'showed', 'demonstrated', 
                                               'improved', 'reduced', 'increased']):
            # Get context around this sentence
            idx = sentences.index(sent)
            context_sents = sentences[max(0, idx-1):min(len(sentences), idx+2)]
            return ' '.join(context_sents)[:300]
    
    return None

In [6]:
def extract_end_to_end(text):
    """
    End-to-End extraction using sentence splitting and intelligent scoring
    """
    results = {}
    
    # Split into sentences
    sentences = split_into_sentences(text)
    
    # Extract each field
    results['population'] = extract_population_end_to_end(text, sentences)
    results['intervention'] = extract_intervention_end_to_end(text, sentences)
    results['comparator'] = extract_comparator_end_to_end(text, sentences)
    results['outcome'] = extract_outcome_end_to_end(text, sentences)
    
    return results

## 4. Load Data and Run Extraction

In [7]:
import pandas as pd

df = pd.read_csv("pico_2_shot_results.csv")

print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

abstracts = df["Text"].tolist()
print(f"Processing {len(abstracts)} abstracts...")

FileNotFoundError: [Errno 2] No such file or directory: 'pico_2_shot_results.csv'

In [8]:
# Run extraction on all abstracts
end_to_end_results = [extract_end_to_end(text) for text in abstracts]

# Preview results
print("\nFirst 3 results:")
for i, result in enumerate(end_to_end_results[:3]):
    print(f"\nAbstract {i+1}:")
    for key, value in result.items():
        print(f"  {key}: {value[:100] if value else 'None'}...")

NameError: name 'abstracts' is not defined

In [9]:
# Save results
results_df = pd.DataFrame(end_to_end_results)
results_df.to_csv("baseline_endtoend_improved_results.csv", index=False)
print("Results saved to baseline_endtoend_improved_results.csv")

NameError: name 'end_to_end_results' is not defined

## 5. Evaluation Against Gold Standard

In [10]:
def pred_to_binary(pred_results):
    """Convert predictions to binary (present/absent)"""
    binary = []
    for r in pred_results:
        binary.append({
            "population": int(r["population"] is not None and r["population"] != "" and len(str(r["population"])) > 3),
            "intervention": int(r["intervention"] is not None and r["intervention"] != "" and len(str(r["intervention"])) > 3),
            "comparator": int(r["comparator"] is not None and r["comparator"] != "" and len(str(r["comparator"])) > 3),
            "outcome": int(r["outcome"] is not None and r["outcome"] != "" and len(str(r["outcome"])) > 3)
        })
    return binary


def gold_to_binary(df):
    """Convert gold labels to binary"""
    gold = []
    for _, row in df.iterrows():
        gold.append({
            "population": int(row.get("Pop_Status", "Incorrect") == "Correct"),
            "intervention": int(row.get("Int_Status", "Incorrect") == "Correct"),
            "outcome": int(row.get("Out_Status", "Incorrect") == "Correct"),
            "comparator": 0  # Not typically in gold set
        })
    return gold

In [11]:
def evaluate(gold, pred):
    """Evaluate predictions against gold standard"""
    fields = ["population", "intervention", "outcome", "comparator"]
    rows = []

    for f in fields:
        y_true = [g[f] for g in gold]
        y_pred = [p[f] for p in pred]

        # Only evaluate if there's variation in ground truth
        if sum(y_true) > 0:
            precision = precision_score(y_true, y_pred, zero_division=0)
            recall = recall_score(y_true, y_pred, zero_division=0)
            f1 = f1_score(y_true, y_pred, zero_division=0)
        else:
            precision = recall = f1 = 0.0

        rows.append({
            "Field": f,
            "Precision": precision,
            "Recall": recall,
            "F1": f1,
        })

    return pd.DataFrame(rows)


gold = gold_to_binary(df)
pred = pred_to_binary(end_to_end_results)

eval_results = evaluate(gold, pred)
print("\nEvaluation Results:")
print(eval_results)

eval_results.to_csv("baseline_endtoend_improved_eval.csv", index=False)

NameError: name 'df' is not defined

## 6. Analysis of Results

In [12]:
# Show coverage statistics
coverage = {
    'population': sum(1 for r in end_to_end_results if r['population'] is not None) / len(end_to_end_results),
    'intervention': sum(1 for r in end_to_end_results if r['intervention'] is not None) / len(end_to_end_results),
    'comparator': sum(1 for r in end_to_end_results if r['comparator'] is not None) / len(end_to_end_results),
    'outcome': sum(1 for r in end_to_end_results if r['outcome'] is not None) / len(end_to_end_results),
}

print("\nCoverage (% of abstracts with extraction):")
for field, cov in coverage.items():
    print(f"  {field}: {cov*100:.1f}%")

NameError: name 'end_to_end_results' is not defined